In [1]:
import pandas as pd
from pandas import read_csv
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OrdinalEncoder, OneHotEncoder, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier 
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, roc_auc_score

In [2]:
class UNSW_NB15:
    def __init__(self):
        self.random_state = 42
        self.scaler = MinMaxScaler()
        self.onehot_encoder = {}
        self.label_encoder = {}
        self.corr = pd.Series()
        self.models = {
            'Decision Tree': DecisionTreeClassifier(random_state = self.random_state), 
            'Random Forest': RandomForestClassifier(random_state = self.random_state, max_depth = 5),
            'K Neighbors': KNeighborsClassifier(n_neighbors = 5),
            'MLP': MLPClassifier(random_state = self.random_state, max_iter = 100, hidden_layer_sizes = (200,)),
            'Naive Bayes': BernoulliNB()
        }
        self.train = None
        self.test = None

    def load_data(self, train_path, test_path):
        #load data
        print("LOADING UNSW_NB15 DATASET...")

        self.train = pd.read_csv(train_path)
        self.test = pd.read_csv(test_path)

        print("LOADING UNSW_NB15 DATASET SUCCESSFULLY!") 

        #overview
        print('=' * 100)
        print("Train and Test data overview")

        print(f"Training data shape: {self.train.shape}")
        print(f"Training data columns: {list(self.train.columns)}")
        print(f"Training data info\n: {self.train.info}")
        print(f"Training data check null:\n {self.train.isnull().sum()}")
        print(f"Training data check duplicate: {self.train.duplicated().sum()}")
        print(f"Training data first 5 columns:\n {self.train.head(5)}")


        print(f"Test data shape: {self.test.shape}")
        print(f"Test data columns: {list(self.test.columns)}")
        print(f"Test data info\n: {self.test.info}")
        print(f"Test data check null:\n {self.test.isnull().sum()}")
        print(f"Test data check duplicate: {self.test.duplicated().sum()}")
        print(f"Test data first 5 columns:\n {self.test.head(5)}")

        #Class distribution
        print('=' * 100)
        print("Class distribution")

        print("Class distribution in Training data ('label'):")
        print(self.train['label'].value_counts())

        print("\nAttack categories distribution in Training Data ('attack_cat'):")
        print(self.train['attack_cat'].value_counts())

        print("\nClass distribution in Testing Data ('label'):")
        print(self.test['label'].value_counts())

        print("\nAttack categories distribution in Testing Data ('attack_cat'):")
        print(self.test['attack_cat'].value_counts())


        #Correlation
        self.corr = (self.train.corr(numeric_only=True)['label'].drop(['label', 'id']).abs().sort_values(ascending=False))    
        print("\nCorrelation between feature and label:")
        print(self.corr)

        #return
        return self.train, self.test


In [3]:
data = UNSW_NB15()

train_path = 'UNSW_NB15_training-set.csv'
test_path = 'UNSW_NB15_testing-set.csv'

raw_train, raw_test = data.load_data(train_path, test_path)

LOADING UNSW_NB15 DATASET...
LOADING UNSW_NB15 DATASET SUCCESSFULLY!
Train and Test data overview
Training data shape: (175341, 45)
Training data columns: ['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']
Training data info
: <bound method DataFrame.info of             id       dur proto service state  spkts  dpkts  sbytes  dbytes  \
0            1  0.121478   tcp       -   FIN      6      4     258     172   
1            2  0.649902   tcp       -   FIN     14     38     734   42014   
2            3  1.623129   tcp       -   

In [4]:
def preprocess(self):
    print("START PREPROCESSING DATA..." + '\n' + '=' * 100)

    #rip 'id'
    print("Remove unnecessary columns...")
    to_drop = ['id']
    self.train_processed = self.train.drop(to_drop, axis = 1)
    self.test_processed = self.test.drop(to_drop, axis = 1)
    print("Completed" + '\n' + '=' * 100)
    """
    #Replace inf value = NaN
    print("Handling -inf and inf values in Train and Test...")
    self.train_processed = self.train_processed.replace([np.inf, -np.inf], np.nan)
    self.test_processed = self.test_processed.replace([np.inf, -np.inf], np.nan)
    print("Completed" + '\n' + '=' * 100)

    #Drop NaN row
    print("Dropping rows that contain NaN value in Train and Test...")
    row_before = len(self.train_processed)
    self.train_processed = self.train_processed.dropna()
    row_after = len(self.train_processed)
    print(f'Train: Dropped {row_before - row_after}, from {row_before} to {row_after}\n')

    row_before = len(self.test_processed)
    self.test_processed = self.test_processed.dropna()
    row_after = len(self.test_processed)
    print(f'Test: Dropped {row_before - row_after}, from {row_before} to {row_after}\n')
    print("Completed" + '\n' + '=' * 100)
    """
    #split x, y
    print("Separate feature and target...")
    features = [col for col in self.train_processed.columns if col not in ['attack_cat', 'label']]

    X_train = self.train_processed[features].copy()
    X_test = self.test_processed[features].copy()
    y_train_binary = self.train_processed['label'].copy()
    y_test_binary = self.test_processed['label'].copy()
    y_train_multi = self.train_processed['attack_cat'].copy()
    y_test_multi = self.test_processed['attack_cat'].copy()
    print("Completed" + '\n' + '=' * 100)

    #Indentify numerical and categorical
    print("Indentify numerical and categorical columns (no target columns)...")
    numerical_col = X_train.select_dtypes(include = [np.number]).columns
    categorical_cols = X_train.select_dtypes(include = [object]).columns

    print(f'Total features: {len(features)}\n')
    print(f'Total numerical features: {len(numerical_col)}\n')
    print(f'Total categorical features: {len(categorical_cols)}\n')
    print("Completed" + '\n' + '=' * 100)

    #onehotencoding
    print('Applying Onehot enconding for categorical features...')
    print(f'X_train shape before Onehot encoding: {X_train.shape}\n')
    for col in categorical_cols:
        ohe = OneHotEncoder(sparse_output=False, handle_unknown = 'ignore')
        #Fit on combined data to ensure consistent encoding
        #combined_col = pd.concat([X_train[col], X_test[col]], ignore_index=True).to_frame()
        #ohe.fit(combined_col.astype(str))
        ohe.fit(X_train[[col]].astype(str))
        ohe_cols = ohe.get_feature_names_out()

        train_onehot = ohe.transform(X_train[[col]].astype(str))
        test_onehot = ohe.transform(X_test[[col]].astype(str))

        train_onehot_df = pd.DataFrame(train_onehot, columns = ohe_cols, index = X_train.index)
        test_onehot_df =  pd.DataFrame(test_onehot, columns = ohe_cols, index = X_test.index)

        X_train = pd.concat([X_train.drop(columns = col), train_onehot_df], axis = 1)
        X_test = pd.concat([X_test.drop(columns = col), test_onehot_df], axis = 1)

        self.onehot_encoder[col] = ohe

    print(f'X_train shape after Onehot encoding: {X_train.shape}\n')
    print("Completed" + '\n' + '=' * 100)

    #Scale
    print('Applying MinMaxScaler for numerical features...')
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()

    X_train_scaled[numerical_col] = self.scaler.fit_transform(X_train_scaled[numerical_col])
    X_test_scaled[numerical_col] = self.scaler.transform(X_test_scaled[numerical_col])
    print("Completed" + '\n' + '=' * 100)

    #Label encoding
    print('Applying Label enconding for target features...')
    self.multi_label_encoder = LabelEncoder()

    combined_multi = pd.concat([y_train_multi, y_test_multi], ignore_index=True)
    self.multi_label_encoder.fit(combined_multi.astype(str))
    self.multi_classes_ = self.multi_label_encoder.classes_.tolist()     

    y_train_multi_encoded = self.multi_label_encoder.transform(y_train_multi.astype(str))
    y_test_multi_encoded = self.multi_label_encoder.transform(y_test_multi.astype(str))
    
    print("Completed" + '\n' + '=' * 100)

    #return
    self.X_train = X_train
    self.X_test = X_test
    self.X_train_scaled = X_train_scaled
    self.X_test_scaled = X_test_scaled
    self.y_train_binary = y_train_binary
    self.y_test_binary = y_test_binary
    self.y_train_multi = y_train_multi_encoded
    self.y_test_multi = y_test_multi_encoded

    print('PREPROCESSING DATA SUCCESSFULLY!') 
    return self.X_train, self.X_test, self.X_train_scaled, self.X_test_scaled, self.y_train_binary, self.y_test_binary, self.y_train_multi, self.y_test_multi

# Add method to processor class
UNSW_NB15.preprocess = preprocess

# Execute 
X_train, X_test, X_train_scaled, X_test_scaled, y_train_binary, y_test_binary, y_train_multi, y_test_multi = data.preprocess()

START PREPROCESSING DATA...
Remove unnecessary columns...
Completed
Separate feature and target...
Completed
Indentify numerical and categorical columns (no target columns)...
Total features: 42

Total numerical features: 39

Total categorical features: 3

Completed
Applying Onehot enconding for categorical features...
X_train shape before Onehot encoding: (175341, 42)

X_train shape after Onehot encoding: (175341, 194)

Completed
Applying MinMaxScaler for numerical features...
Completed
Applying Label enconding for target features...
Completed
PREPROCESSING DATA SUCCESSFULLY!


In [5]:
def Feature_Selection(self):
    print("START SELECTING DATA..." + '\n' + '=' * 100)

    #get correlation rank
    print('Calculating correlation matrix...')
    features = self.train_processed.select_dtypes(include = [np.number])
    features = features.drop(['label'], axis = 1)
    
    corr_matrix = features.corr()
    np.fill_diagonal(corr_matrix.values, np.nan)
    avr_corr = corr_matrix.mean(skipna = True)
    avr_corr_sorted = avr_corr.sort_values(ascending = False)
    print("Completed" + '\n' + '=' * 100)

    #get top K
    print('Getting top K features...')
    col_4 = avr_corr_sorted.head(4).index.tolist()
    col_8 = avr_corr_sorted.head(8).index.tolist()
    col_16 = avr_corr_sorted.head(16).index.tolist()
    col_20 = avr_corr_sorted.head(20).index.tolist()

    self.X_train_selection_4 = self.X_train[col_4].copy()
    self.X_test_selection_4 = self.X_test[col_4].copy()
    self.X_train_selection_8 = self.X_train[col_8].copy()
    self.X_test_selection_8 = self.X_test[col_8].copy()
    self.X_train_selection_16 = self.X_train[col_16].copy()
    self.X_test_selection_16 = self.X_test[col_16].copy()
    self.X_train_selection_20 = self.X_train[col_20].copy()
    self.X_test_selection_20 = self.X_test[col_20].copy()
    print("Completed" + '\n' + '=' * 100)

    #retutn
    print('SELECTING DATA SUCCESSFULLY!')
    return (avr_corr_sorted, 
            self.X_train_selection_4, 
            self.X_test_selection_4, 
            self.X_train_selection_8, 
            self.X_test_selection_8, 
            self.X_train_selection_16, 
            self.X_test_selection_16,
            self.X_train_selection_20,
            self.X_test_selection_20)

# Add method to processor class
UNSW_NB15.feature_selection = Feature_Selection

#Execute
avr_corr_list, X_train_selection_4, X_test_selection_4, X_train_selection_8, X_test_selection_8, X_train_selection_16, X_test_selection_16, X_train_selection_20, X_test_selection_20 = data.feature_selection()


START SELECTING DATA...
Calculating correlation matrix...
Completed
Getting top K features...
Completed
SELECTING DATA SUCCESSFULLY!


In [6]:
def Feature_Extraction(self):
    print("START EXTRACTING DATA..." + '\n' + '=' * 100)
    original_feature = self.X_train_scaled.shape[1]
    self.PCA_4 = PCA(n_components=4, random_state=self.random_state)
    self.PCA_8 = PCA(n_components=8, random_state=self.random_state)
    self.PCA_16 = PCA(n_components=16, random_state=self.random_state)
    self.PCA_20 = PCA(n_components=20, random_state=self.random_state)

    #4
    print("Extracting to 4 features...")
    X_train_extraction_4 = self.X_train_scaled.copy()
    X_test_extraction_4 = self.X_test_scaled.copy()

    X_train_extraction_4 = pd.DataFrame(self.PCA_4.fit_transform(X_train_extraction_4))
    X_test_extraction_4 = pd.DataFrame(self.PCA_4.transform(X_test_extraction_4))

    print(f'Extracted {original_feature} features to {X_train_extraction_4.shape[1]} features')
    print("Completed" + '\n' + '=' * 100)

    #8
    print("Extracting to 8 features...")
    X_train_extraction_8 = self.X_train_scaled.copy()
    X_test_extraction_8 = self.X_test_scaled.copy()
 
    X_train_extraction_8 = pd.DataFrame(self.PCA_8.fit_transform(X_train_extraction_8))
    X_test_extraction_8 = pd.DataFrame(self.PCA_8.transform(X_test_extraction_8))

    print(f'Extracted {original_feature} features to {X_train_extraction_8.shape[1]} features')
    print("Completed" + '\n' + '=' * 100)

    #16
    print("Extracting to 16 features...")
    X_train_extraction_16 = self.X_train_scaled.copy()
    X_test_extraction_16 = self.X_test_scaled.copy()

    X_train_extraction_16 = pd.DataFrame(self.PCA_16.fit_transform(X_train_extraction_16))
    X_test_extraction_16 = pd.DataFrame(self.PCA_16.transform(X_test_extraction_16))

    print(f'Extracted {original_feature} features to {X_train_extraction_16.shape[1]} features')
    print("Completed" + '\n' + '=' * 100)

    #20
    print("Extracting to 20 features...")
    X_train_extraction_20 = self.X_train_scaled.copy()
    X_test_extraction_20= self.X_test_scaled.copy()

    X_train_extraction_20 = pd.DataFrame(self.PCA_20.fit_transform(X_train_extraction_20))
    X_test_extraction_20 = pd.DataFrame(self.PCA_20.transform(X_test_extraction_20))

    print(f'Extracted {original_feature} features to {X_train_extraction_20.shape[1]} features')
    print("Completed" + '\n' + '=' * 100)

    #return
    self.X_train_extraction_4 = X_train_extraction_4
    self.X_test_extraction_4 = X_test_extraction_4
    self.X_train_extraction_8 = X_train_extraction_8
    self.X_test_extraction_8 = X_test_extraction_8
    self.X_train_extraction_16 = X_train_extraction_16
    self.X_test_extraction_16 = X_test_extraction_16
    self.X_train_extraction_20 = X_train_extraction_20
    self.X_test_extraction_20 = X_test_extraction_20

    print('EXTRACTING DATA SUCCESSFULLY!')
    return (self.X_train_extraction_4, 
            self.X_test_extraction_4, 
            self.X_train_extraction_8, 
            self.X_test_extraction_8, 
            self.X_train_extraction_16, 
            self.X_test_extraction_16, 
            self.X_train_extraction_20,
            self.X_test_extraction_20)
    
# Add method to processor class    
UNSW_NB15.feature_extraction = Feature_Extraction

#Execute
X_train_extraction_4, X_test_extraction_4, X_train_extraction_8, X_test_extraction_8, X_train_extraction_16, X_test_extraction_16, X_train_extraction_20, X_test_extraction_20 = data.feature_extraction()

START EXTRACTING DATA...
Extracting to 4 features...
Extracted 194 features to 4 features
Completed
Extracting to 8 features...
Extracted 194 features to 8 features
Completed
Extracting to 16 features...
Extracted 194 features to 16 features
Completed
Extracting to 20 features...
Extracted 194 features to 20 features
Completed
EXTRACTING DATA SUCCESSFULLY!


In [ ]:
def Score(labels, preds):
    precision = precision_score(labels, preds, average = 'weighted') 
    recall = recall_score(labels, preds, average = 'weighted')
    f1 = f1_score(labels, preds, average = 'weighted')

    print(f'Precision: {precision * 100:.2f}')
    print(f'Recall: {recall * 100:.2f}')
    print(f'F1-score: {f1 * 100:.2f}')

#Add method to processor class
UNSW_NB15.score = staticmethod(Score)

In [8]:
def Model(self, x_train_final, y_train_final, x_test_final, y_test_final, type):
    #train
    for id, model_ in self.models.items():
        print(model_)
        model_.fit(x_train_final, y_train_final)
        y_predict_final = model_.predict(x_test_final)
        data.score(y_test_final, y_predict_final)
        print('_' * 50)
        """
        if(type == 0):
            print(classification_report(y_test_final, y_predict_final, digits = 4))
        else:
            print(classification_report(y_test_final, y_predict_final, digits = 4, target_names = self.multi_classes_))
        """
        
# Add method to processor class
UNSW_NB15.model = Model


In [9]:
def Training(self):
    print("START TRAINING..." + '\n' + '=' * 100)

    #BINARY
    print("BINARY CLASSIFICATION" + '\n' + '_' * 80)
    
    #K = 4
    print('K = 4' + '\n' + '_' * 60)

    print('Selection method result:' + '\n' + '.' + '\n' + '.')
    data.model(self.X_train_selection_4, self.y_train_binary, self.X_test_selection_4, self.y_test_binary, 0)
    print('_' * 50)

    print('Extraction method result:' + '\n' + '.' + '\n' + '.')
    data.model(self.X_train_extraction_4, self.y_train_binary, self.X_test_extraction_4, self.y_test_binary, 0)
    print('_' * 50)

    #K = 8
    print('K = 8' + '\n'+ '_' * 50)

    print('Selection method result:' + '\n' + '.' + '\n' + '.')
    data.model(self.X_train_selection_8, self.y_train_binary, self.X_test_selection_8, self.y_test_binary, 0)
    print('_' * 40)

    print('Extraction method result:' + '\n' + '.' + '\n' + '.')
    data.model(self.X_train_extraction_8, self.y_train_binary, self.X_test_extraction_8, self.y_test_binary, 0)
    print('_' * 50)

    #K = 16
    print('K = 16' + '\n' + '_' * 50)

    print('Selection method result:' + '\n' + '.' + '\n' + '.')
    data.model(self.X_train_selection_16, self.y_train_binary, self.X_test_selection_16, self.y_test_binary, 0)
    print('_' * 40)
    
    print('Extraction method result:' + '\n' + '.' + '\n' + '.')
    data.model(self.X_train_extraction_16, self.y_train_binary, self.X_test_extraction_16, self.y_test_binary, 0)
    print('_' * 50)
    
    #K = 20
    print('K = 20' + '\n' + '_' * 50)

    print('Selection method result:' + '\n' + '.' + '\n' + '.')
    data.model(self.X_train_selection_20, self.y_train_binary, self.X_test_selection_20, self.y_test_binary, 0)
    print('_' * 40)
    
    print('Extraction method result:' + '\n' + '.' + '\n' + '.')
    data.model(self.X_train_extraction_20, self.y_train_binary, self.X_test_extraction_20, self.y_test_binary, 0)
    print('\n' + '=' * 100)
    
    #MULTICLASS
    print("MULTICLASS CLASSIFICATION" + '\n' + '_' * 80)
    
    #K = 4
    print('K = 4' + '\n' + '_' * 60)

    print('Selection method result:' + '\n' + '.' + '\n' + '.')
    data.model(self.X_train_selection_4, self.y_train_multi, self.X_test_selection_4, self.y_test_multi, 1)
    print('_' * 40)

    print('Extraction method result:' + '\n' + '.' + '\n' + '.')
    data.model(self.X_train_extraction_4, self.y_train_multi, self.X_test_extraction_4, self.y_test_multi, 1)
    print('_' * 50)

    #K = 8
    print('K = 8' + '\n'+ '_' * 50)

    print('Selection method result:' + '\n' + '.' + '\n' + '.')
    data.model(self.X_train_selection_8, self.y_train_multi, self.X_test_selection_8, self.y_test_multi, 1)
    print('_' * 40)

    print('Extraction method result:' + '\n' + '.' + '\n' + '.')
    data.model(self.X_train_extraction_8, self.y_train_multi, self.X_test_extraction_8, self.y_test_multi, 1)
    print('_' * 50)

    #K = 16
    print('K = 16' + '\n' + '_' * 50)

    print('Selection method result:' + '\n' + '.' + '\n' + '.')
    data.model(self.X_train_selection_16, self.y_train_multi, self.X_test_selection_16, self.y_test_multi, 1)
    print('_' * 40)

    print('Extraction method result:' + '\n' + '.' + '\n' + '.')
    data.model(self.X_train_extraction_16, self.y_train_multi, self.X_test_extraction_16, self.y_test_multi, 1)
    print('_' * 50)
    
    #K = 20
    print('K = 20' + '\n' + '_' * 50)

    print('Selection method result:' + '\n' + '.' + '\n' + '.')
    data.model(self.X_train_selection_20, self.y_train_multi, self.X_test_selection_20, self.y_test_multi, 1)
    print('_' * 40)

    print('Extraction method result:' + '\n' + '.' + '\n' + '.')
    data.model(self.X_train_extraction_20, self.y_train_multi, self.X_test_extraction_20, self.y_test_multi, 1)
    print('\n' + '=' * 100)

    print('TRAINING SUCCESSFULLY!')
# Add method to processor class
UNSW_NB15.training = Training

data.training()

START TRAINING...
BINARY CLASSIFICATION
________________________________________________________________________________
K = 4
____________________________________________________________
Selection method result:
.
.
DecisionTreeClassifier(random_state=42)
Precision: 0.8414
Recall: 0.7991
F1-score: 0.7876
__________________________________________________
RandomForestClassifier(max_depth=5, random_state=42)
Precision: 0.7798
Recall: 0.7518
F1-score: 0.7388
__________________________________________________
KNeighborsClassifier()
Precision: 0.5256
Recall: 0.4806
F1-score: 0.4289
__________________________________________________
MLPClassifier(hidden_layer_sizes=(200,), max_iter=100, random_state=42)
Precision: 0.7530
Recall: 0.7445
F1-score: 0.7376
__________________________________________________
BernoulliNB()
Precision: 0.7548
Recall: 0.7363
F1-score: 0.7359
__________________________________________________
__________________________________________________
Extraction method result:

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


Precision: 0.8630
Recall: 0.8274
F1-score: 0.8194
__________________________________________________
BernoulliNB()
Precision: 0.7206
Recall: 0.6994
F1-score: 0.6818
__________________________________________________
__________________________________________________
K = 16
__________________________________________________
Selection method result:
.
.
DecisionTreeClassifier(random_state=42)
Precision: 0.8747
Recall: 0.8659
F1-score: 0.8638
__________________________________________________
RandomForestClassifier(max_depth=5, random_state=42)
Precision: 0.8578
Recall: 0.8087
F1-score: 0.7971
__________________________________________________
KNeighborsClassifier()
Precision: 0.7966
Recall: 0.7835
F1-score: 0.7773
__________________________________________________
MLPClassifier(hidden_layer_sizes=(200,), max_iter=100, random_state=42)
Precision: 0.8162
Recall: 0.7799
F1-score: 0.7779
__________________________________________________
BernoulliNB()
Precision: 0.6890
Recall: 0.6834
F1-scor

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


Precision: 0.8569
Recall: 0.8194
F1-score: 0.8106
__________________________________________________
BernoulliNB()
Precision: 0.7813
Recall: 0.7551
F1-score: 0.7429
__________________________________________________
__________________________________________________
K = 20
__________________________________________________
Selection method result:
.
.
DecisionTreeClassifier(random_state=42)
Precision: 0.8771
Recall: 0.8658
F1-score: 0.8633
__________________________________________________
RandomForestClassifier(max_depth=5, random_state=42)
Precision: 0.8577
Recall: 0.8086
F1-score: 0.7969
__________________________________________________
KNeighborsClassifier()
Precision: 0.8018
Recall: 0.7815
F1-score: 0.7732
__________________________________________________
MLPClassifier(hidden_layer_sizes=(200,), max_iter=100, random_state=42)
Precision: 0.6882
Recall: 0.6877
F1-score: 0.6819
__________________________________________________
BernoulliNB()
Precision: 0.6891
Recall: 0.6835
F1-scor

...

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Precision: 0.7812
Recall: 0.6673
F1-score: 0.6810
__________________________________________________
KNeighborsClassifier()
Precision: 0.7850
Recall: 0.7074
F1-score: 0.7364
__________________________________________________
MLPClassifier(hidden_layer_sizes=(200,), max_iter=100, random_state=42)


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


Precision: 0.7969
Recall: 0.7415
F1-score: 0.7453
__________________________________________________
BernoulliNB()
Precision: 0.7380
Recall: 0.6025
F1-score: 0.6422
__________________________________________________

TRAINING SUCCESSFULLY!


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


GRIDSEARCH

In [10]:
x_train_ = X_train_selection_20
y_train_ = y_train_binary
x_test_ = X_test_selection_20
y_test_ = y_test_binary

In [13]:
model = DecisionTreeClassifier(random_state = 42)

param_grid = {
    "criterion": ["gini", "entropy", "log_loss"],   # Hàm đo độ phân chia
    "max_depth": [None, 5, 10, 20, 50],             # Độ sâu tối đa của cây
    "min_samples_split": [2, 5, 10, 20],            # Số mẫu tối thiểu để tách nhánh
    "min_samples_leaf": [1, 2, 5, 10],              # Số mẫu tối thiểu ở một lá
}

grid = GridSearchCV(estimator = model, param_grid = param_grid, scoring = 'f1_weighted', cv = 4, verbose = 0)
grid.fit(x_train_, y_train_)
y_predict_ = grid.predict(x_test_)
precision = precision_score(y_test_, y_predict_, average = 'weighted')
recall = recall_score(y_test_, y_predict_, average = 'weighted')
f1 = f1_score(y_test_, y_predict_, average = 'weighted')

print(f'Precision: {precision * 100:.2f}')
print(f'Recall: {recall * 100:.2f}')
print(f'F1-score: {f1 * 100:.2f}')

best_model = grid.best_estimator_
print(f'BEST MODEL: {best_model}')
print(f'BEST SCORE: {grid.best_score_ * 100:.2f}')
print(f'BEST PARAMS: {grid.best_params_}')

Precision: 85.84
Recall: 81.69
F1-score: 80.72
BEST MODEL: DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=42)
BEST SCORE: 90.23
BEST PARAMS: {'criterion': 'entropy', 'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2}


In [14]:
model = RandomForestClassifier(random_state = 42, n_jobs = -1)

param_grid = {
    "n_estimators": [100, 200, 300],            # số lượng cây trong rừng
    "criterion": ["gini", "entropy", "log_loss"], # hàm đánh giá độ phân chia
    "max_depth": [None, 10, 20],            # độ sâu tối đa của cây
    "min_samples_split": [2, 5, 7],            # số mẫu tối thiểu để tách nhánh
    "min_samples_leaf": [1, 2, 3],              # số mẫu tối thiểu ở một lá
}

grid = GridSearchCV(estimator = model, param_grid = param_grid, scoring = 'f1_weighted', cv = 4, verbose = 0)
grid.fit(x_train_, y_train_)
y_predict_ = grid.predict(x_test_)
precision = precision_score(y_test_, y_predict_, average = 'weighted')
recall = recall_score(y_test_, y_predict_, average = 'weighted')
f1 = f1_score(y_test_, y_predict_, average = 'weighted')

print(f'Precision: {precision * 100:.2f}')
print(f'Recall: {recall * 100:.2f}')
print(f'F1-score: {f1 * 100:.2f}')

best_model = grid.best_estimator_
print(f'BEST MODEL: {best_model}')
print(f'BEST SCORE: {grid.best_score_ * 100:.2f}')
print(f'BEST PARAMS: {grid.best_params_}')

Precision: 88.60
Recall: 86.37
F1-score: 85.96
BEST MODEL: RandomForestClassifier(min_samples_leaf=3, n_estimators=200, n_jobs=-1,
                       random_state=42)
BEST SCORE: 88.74
BEST PARAMS: {'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 3, 'min_samples_split': 2, 'n_estimators': 200}


In [15]:
model = KNeighborsClassifier(n_jobs = -1)

param_grid = {
    "n_neighbors": [3, 5, 7, 9, 11],           # số lượng láng giềng
    "weights": ["uniform", "distance"],        # cách tính trọng số
    "metric": ["euclidean", "manhattan", "minkowski"], # hàm khoảng cách
    "p": [1, 2]                                # chỉ số p cho Minkowski (1=Manhattan, 2=Euclidean)
}

grid = GridSearchCV(estimator = model, param_grid = param_grid, scoring = 'f1_weighted', cv = 4, verbose = 0)
grid.fit(x_train_, y_train_)
y_predict_ = grid.predict(x_test_)
precision = precision_score(y_test_, y_predict_, average = 'weighted')
recall = recall_score(y_test_, y_predict_, average = 'weighted')
f1 = f1_score(y_test_, y_predict_, average = 'weighted')

print(f'Precision: {precision * 100:.2f}')
print(f'Recall: {recall * 100:.2f}')
print(f'F1-score: {f1 * 100:.2f}')

best_model = grid.best_estimator_
print(f'BEST MODEL: {best_model}')
print(f'BEST SCORE: {grid.best_score_ * 100:.2f}')
print(f'BEST PARAMS: {grid.best_params_}')

Precision: 80.90
Recall: 78.25
F1-score: 77.27
BEST MODEL: KNeighborsClassifier(metric='manhattan', n_jobs=-1, n_neighbors=11, p=1)
BEST SCORE: 85.95
BEST PARAMS: {'metric': 'manhattan', 'n_neighbors': 11, 'p': 1, 'weights': 'uniform'}


In [ ]:
model = MLPClassifier(random_state = 42)

param_grid = {
    "hidden_layer_sizes": [(100,), (100, 50), (50, 50, 50)],  # cấu trúc hidden layers
    "activation": ["relu", "tanh"],  # hàm kích hoạt
    "solver": ["adam", "sgd"],          # thuật toán tối ưu
    "alpha": [0.0001, 0.001],              # hệ số regularization (L2 penalty)
    "learning_rate": ["constant", "adaptive"],   # cách thay đổi learning rate
    "max_iter": [200, 300]                       # số vòng lặp tối đa
}

grid = GridSearchCV(estimator = model, param_grid = param_grid, scoring = 'f1_weighted', cv = 4, verbose = 0)
grid.fit(x_train_, y_train_)
y_predict_ = grid.predict(x_test_)
precision = precision_score(y_test_, y_predict_, average = 'weighted')
recall = recall_score(y_test_, y_predict_, average = 'weighted')
f1 = f1_score(y_test_, y_predict_, average = 'weighted')

print(f'Precision: {precision * 100:.2f}')
print(f'Recall: {recall * 100:.2f}')
print(f'F1-score: {f1 * 100:.2f}')

best_model = grid.best_estimator_
print(f'BEST MODEL: {best_model}')
print(f'BEST SCORE: {grid.best_score_ * 100:.2f}')
print(f'BEST PARAMS: {grid.best_params_}')

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\extmath.py:203: RuntimeWarning: overflow encountered in matmu

In [ ]:
model = BernoulliNB()

param_grid = {
    "alpha": [0.01, 0.1, 0.5, 1.0, 5.0],   # hệ số smoothing Laplace
    "binarize": [None, 0.0, 0.5, 1.0],          # ngưỡng để chuyển thành dữ liệu nhị phân
    "fit_prior": [True, False]                  # có học prior từ dữ liệu hay không
}
grid = GridSearchCV(estimator = model, param_grid = param_grid, scoring = 'f1_weighted', cv = 4, verbose = 0)
grid.fit(x_train_, y_train_)
y_predict_ = grid.predict(x_test_)
precision = precision_score(y_test_, y_predict_, average = 'weighted')
recall = recall_score(y_test_, y_predict_, average = 'weighted')
f1 = f1_score(y_test_, y_predict_, average = 'weighted')

print(f'Precision: {precision * 100:.2f}')
print(f'Recall: {recall * 100:.2f}')
print(f'F1-score: {f1 * 100:.2f}')

best_model = grid.best_estimator_
print(f'BEST MODEL: {best_model}')
print(f'BEST SCORE: {grid.best_score_ * 100:.2f}')
print(f'BEST PARAMS: {grid.best_params_}')